In [2]:
# Optional: install dependencies (uncomment if needed)
# !pip install torch torchvision scikit-learn tqdm
import os
os.environ.setdefault('MKL_THREADING_LAYER', 'GNU')
import time
import math
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

In [4]:
# Reproducible settings and synthetic data generator
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

DEFAULT_SAMPLES = 10_000
N_SAMPLES = int(os.environ.get('MM_DEMO_MAX_SAMPLES', DEFAULT_SAMPLES))
if N_SAMPLES != DEFAULT_SAMPLES:
    print(f'Overriding default sample count -> {N_SAMPLES}')

def spectral_field(num_samples, dim, alpha=1.5, freq_scale=0.05, noise_std=1.0, rng=None):
    """Generate smooth correlated static signals by filtering white noise in Fourier space."""
    rng = rng or np.random.default_rng()
    freqs = np.fft.rfftfreq(dim)
    safe_freqs = np.maximum(freqs, 1e-4)
    filt = 1.0 / (1.0 + (safe_freqs / freq_scale) ** alpha)
    coeff_real = rng.normal(scale=noise_std, size=(num_samples, freqs.size))
    coeff_imag = rng.normal(scale=noise_std, size=(num_samples, freqs.size))
    coeffs = (coeff_real + 1j * coeff_imag) * filt
    coeffs[:, 0].imag = 0.0  # enforce real signal
    fields = np.fft.irfft(coeffs, n=dim, axis=1)
    fields = fields.astype(np.float32)
    fields -= fields.mean(axis=1, keepdims=True)
    fields /= fields.std(axis=1, keepdims=True) + 1e-6
    return fields

def stochastic_oscillator(num_samples, dim, harmonics=4, freq_range=(0.2, 6.0), noise_std=0.35, rng=None):
    """Simulate bursty temporal telemetry via random harmonic oscillators with injected noise."""
    rng = rng or np.random.default_rng()
    time = np.linspace(0.0, 1.0, dim, dtype=np.float32)
    signals = np.zeros((num_samples, dim), dtype=np.float32)
    for h in range(1, harmonics + 1):
        freqs = rng.uniform(freq_range[0], freq_range[1], size=(num_samples, 1))
        phases = rng.uniform(0.0, 2 * np.pi, size=(num_samples, 1))
        amps = rng.normal(loc=1.0 / h, scale=0.15, size=(num_samples, 1))
        oscillation = np.sin(2 * np.pi * (freqs * time + phases))
        signals += (amps * oscillation).astype(np.float32)
    envelopes = rng.lognormal(mean=0.0, sigma=0.25, size=(num_samples, 1)).astype(np.float32)
    signals *= envelopes
    noise = rng.normal(scale=noise_std, size=signals.shape).astype(np.float32)
    signals += noise
    signals -= signals.mean(axis=1, keepdims=True)
    signals /= signals.std(axis=1, keepdims=True) + 1e-6
    return signals

def network_feature_block(num_samples, dim, base_rate=1.0, tail_strength=0.25, rng=None):
    """Create heavy-tailed network traffic descriptors with occasional beaconing bursts."""
    rng = rng or np.random.default_rng()
    log_means = rng.normal(loc=base_rate, scale=0.35, size=(num_samples, dim))
    log_stds = rng.uniform(0.15, 0.5, size=(num_samples, dim))
    counts = rng.lognormal(mean=log_means, sigma=log_stds).astype(np.float32)
    bursts = rng.binomial(1, tail_strength, size=(num_samples, dim))
    burst_magnitudes = rng.lognormal(mean=3.5, sigma=0.4, size=(num_samples, dim)).astype(np.float32)
    counts += bursts * burst_magnitudes
    window = np.linspace(0.2, 1.0, 5, dtype=np.float32)
    beacon = rng.normal(scale=0.5, size=(num_samples, dim)).astype(np.float32)
    beacon[:, :5] += (counts[:, :5] * window).mean(axis=1, keepdims=True)
    counts += beacon
    counts = np.clip(counts, a_min=0.0, a_max=None)
    counts = np.log1p(counts)
    counts -= counts.mean(axis=1, keepdims=True)
    counts /= counts.std(axis=1, keepdims=True) + 1e-6
    return counts.astype(np.float32)

def fast_cfg_api_features(num_samples, dim, latent_dim=384, api_bins=64, rng=None):
    """Approximate CFG/API graph descriptors via vectorized latent histograms and message passing surrogates."""
    rng = rng or np.random.default_rng()
    branchiness = rng.beta(2.0, 3.0, size=(num_samples, latent_dim)).astype(np.float32)
    loop_complexity = rng.normal(loc=0.0, scale=1.0, size=(num_samples, latent_dim)).astype(np.float32)
    depth_profile = rng.gamma(shape=2.5, scale=0.8, size=(num_samples, latent_dim)).astype(np.float32)
    api_hist = rng.dirichlet(alpha=np.ones(api_bins), size=num_samples).astype(np.float32)
    api_features = api_hist @ rng.normal(scale=0.6, size=(api_bins, latent_dim)).astype(np.float32)
    fused = np.concatenate([branchiness, loop_complexity, depth_profile, api_features], axis=1)
    if fused.shape[1] >= dim:
        features = fused[:, :dim]
    else:
        repeat = int(np.ceil(dim / fused.shape[1]))
        features = np.tile(fused, (1, repeat))[:, :dim]
    features -= features.mean(axis=1, keepdims=True)
    features /= features.std(axis=1, keepdims=True) + 1e-6
    return features.astype(np.float32)

static_dim = 2048
dynamic_dim = 1536
graphical_dim = 3072
network_dim = 2048
NUM_CLASSES = 2

X_static = spectral_field(N_SAMPLES, static_dim, alpha=1.6, freq_scale=0.03, noise_std=0.6, rng=rng)
X_dynamic = stochastic_oscillator(N_SAMPLES, dynamic_dim, harmonics=6, freq_range=(0.5, 5.5), noise_std=0.35, rng=rng)
X_graphical = fast_cfg_api_features(N_SAMPLES, graphical_dim, latent_dim=384, api_bins=64, rng=rng)
X_network = network_feature_block(N_SAMPLES, network_dim, base_rate=0.9, tail_strength=0.3, rng=rng)

N_FAMILIES = 1500
family_labels = rng.integers(0, N_FAMILIES, size=N_SAMPLES)
family_offsets = {
    'static': rng.normal(scale=0.6, size=(N_FAMILIES, static_dim)).astype(np.float32),
    'dynamic': rng.normal(scale=0.5, size=(N_FAMILIES, dynamic_dim)).astype(np.float32),
    'graphical': rng.normal(scale=0.4, size=(N_FAMILIES, graphical_dim)).astype(np.float32),
    'network': rng.normal(scale=0.7, size=(N_FAMILIES, network_dim)).astype(np.float32),
}
X_static += 0.08 * family_offsets['static'][family_labels]
X_dynamic += 0.07 * family_offsets['dynamic'][family_labels]
X_graphical += 0.05 * family_offsets['graphical'][family_labels]
X_network += 0.09 * family_offsets['network'][family_labels]

static_w = rng.normal(scale=0.09, size=static_dim).astype(np.float32)
dynamic_w = rng.normal(scale=0.09, size=dynamic_dim).astype(np.float32)
graph_w = rng.normal(scale=0.08, size=graphical_dim).astype(np.float32)
network_w = rng.normal(scale=0.1, size=network_dim).astype(np.float32)
family_risk = rng.normal(loc=0.0, scale=0.5, size=N_FAMILIES).astype(np.float32)

network_beacon = X_network[:, : network_dim // 4].mean(axis=1)
dynamic_entropy = X_dynamic[:, dynamic_dim // 6 : dynamic_dim // 3].std(axis=1)
graph_density = X_graphical[:, 500:1000].mean(axis=1)
static_contrast = X_static[:, ::16].mean(axis=1)

latent_score = (
    (X_static @ static_w) / np.sqrt(static_dim)
    + (X_dynamic @ dynamic_w) / np.sqrt(dynamic_dim)
    + (X_graphical @ graph_w) / np.sqrt(graphical_dim)
    + (X_network @ network_w) / np.sqrt(network_dim)
)
latent_score += 1.3 * network_beacon + 1.0 * dynamic_entropy
latent_score += 0.7 * graph_density + 0.6 * static_contrast
latent_score += family_risk[family_labels]
latent_score += rng.normal(scale=0.3, size=N_SAMPLES)
probs = 1.0 / (1.0 + np.exp(-latent_score))

threshold = np.median(probs)
y = (probs >= threshold).astype(np.int64)
print('Class balance -> malicious ratio:', round(y.mean(), 3))

ids = np.arange(N_SAMPLES)
train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
train_ids, val_ids = train_test_split(train_ids, test_size=0.2, random_state=RANDOM_SEED, stratify=y[train_ids])

feature_blocks = {
    'static': X_static,
    'dynamic': X_dynamic,
    'graphical': X_graphical,
    'network': X_network,
}
scalers = {}
for name, feat in feature_blocks.items():
    scaler = StandardScaler()
    scaler.fit(feat[train_ids])
    feature_blocks[name] = scaler.transform(feat).astype(np.float32)
    scalers[name] = scaler

X_static = feature_blocks['static']
X_dynamic = feature_blocks['dynamic']
X_graphical = feature_blocks['graphical']
X_network = feature_blocks['network']

def slice_by_ids(ids_array):
    return dict(
        static=X_static[ids_array],
        dynamic=X_dynamic[ids_array],
        graphical=X_graphical[ids_array],
        network=X_network[ids_array],
        labels=y[ids_array]
)

train_data = slice_by_ids(train_ids)
val_data = slice_by_ids(val_ids)
test_data = slice_by_ids(test_ids)

print('Train/Val/Test sizes:', len(train_ids), len(val_ids), len(test_ids))

Class balance -> malicious ratio: 0.5
Train/Val/Test sizes: 6400 1600 2000
Train/Val/Test sizes: 6400 1600 2000


In [5]:
# Dataset and DataLoader wrappers
class MalwareDataset(Dataset):
    """Simple mapping from numpy feature blocks into torch tensors per modality."""
    def __init__(self, data_dict):
        self.static = torch.from_numpy(data_dict['static'])
        self.dynamic = torch.from_numpy(data_dict['dynamic'])
        self.graphical = torch.from_numpy(data_dict['graphical'])
        self.network = torch.from_numpy(data_dict['network'])
        self.labels = torch.from_numpy(data_dict['labels']).long()
    def __len__(self):
        return self.labels.shape[0]
    def __getitem__(self, idx):
        return dict(
            static=self.static[idx],
            dynamic=self.dynamic[idx],
            graphical=self.graphical[idx],
            network=self.network[idx],
            label=self.labels[idx]
)

batch_size = 256
train_ds = MalwareDataset(train_data)
val_ds = MalwareDataset(val_data)
test_ds = MalwareDataset(test_data)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print('Sample batch shapes:')
batch = next(iter(train_loader))
print('static', batch['static'].shape)
print('dynamic', batch['dynamic'].shape)
print('graphical', batch['graphical'].shape)
print('network', batch['network'].shape)

Sample batch shapes:
static torch.Size([256, 2048])
dynamic torch.Size([256, 1536])
graphical torch.Size([256, 3072])
network torch.Size([256, 2048])
static torch.Size([256, 2048])
dynamic torch.Size([256, 1536])
graphical torch.Size([256, 3072])
network torch.Size([256, 2048])


In [24]:
# Model helpers with LLM/VLM-inspired blocks
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, seq_len, device, offset=0):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype) + offset
        freqs = torch.einsum('i , j -> i j', t, self.inv_freq)
        return freqs.cos()[None, None, :, :], freqs.sin()[None, None, :, :]


def apply_rotary(q, k, cos, sin):
    q1, q2 = q[..., ::2], q[..., 1::2]
    k1, k2 = k[..., ::2], k[..., 1::2]
    q_rot = torch.cat([q1 * cos - q2 * sin, q1 * sin + q2 * cos], dim=-1)
    k_rot = torch.cat([k1 * cos - k2 * sin, k1 * sin + k2 * cos], dim=-1)
    return q_rot, k_rot


class LoRAAdapter(nn.Module):
    def __init__(self, dim, rank=16, alpha=16):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.lora_up = nn.Linear(dim, rank, bias=False)
        self.lora_down = nn.Linear(rank, dim, bias=False)
        nn.init.kaiming_uniform_(self.lora_up.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_down.weight)

    def forward(self, x):
        return x + self.lora_down(self.lora_up(x)) * self.scaling


class PatchEmbed(nn.Module):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(patch_size, embed_dim)

    def forward(self, x):
        bsz, dim = x.shape
        new_dim = int(math.ceil(dim / self.patch_size) * self.patch_size)
        if new_dim != dim:
            pad = new_dim - dim
            x = F.pad(x, (0, pad))
        x = x.view(bsz, -1, self.patch_size)
        return self.proj(x)


class FlashMultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1, rope=None):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout
        self.rope = rope

    def forward(self, x, attn_mask=None, cache=None, use_cache=False, layer_idx=0):
        bsz, seq_len, _ = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        if self.rope is not None:
            offset = 0 if cache is None else cache.get('k', torch.empty(0, device=x.device)).shape[-2]
            cos, sin = self.rope(seq_len, x.device, offset=offset)
            q, k = apply_rotary(q, k, cos, sin)
        if use_cache and cache is not None and 'k' in cache:
            k = torch.cat([cache['k'], k], dim=-2)
            v = torch.cat([cache['v'], v], dim=-2)
        attn_out = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0
        )
        new_cache = None
        if use_cache:
            new_cache = {'k': k.detach(), 'v': v.detach()}
        attn_out = attn_out.transpose(1, 2).contiguous().view(bsz, seq_len, self.embed_dim)
        return self.out_proj(attn_out), new_cache


class MoEFeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_experts=4, top_k=2, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, embed_dim),
            )
            for _ in range(num_experts)
        ])
        self.gate = nn.Linear(embed_dim, num_experts)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gate_logits = self.gate(x)
        probs = torch.softmax(gate_logits, dim=-1)
        topk = torch.topk(probs, k=self.top_k, dim=-1)
        out = torch.zeros_like(x)
        for i in range(self.top_k):
            expert_idx = topk.indices[..., i].unsqueeze(-1)
            expert_weight = topk.values[..., i].unsqueeze(-1)
            for idx in range(self.num_experts):
                mask = (expert_idx.squeeze(-1) == idx)
                if mask.any():
                    masked_x = x[mask]
                    expert_res = self.experts[idx](masked_x)
                    out[mask] = out[mask] + expert_res * expert_weight[mask]
        return self.dropout(out)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, rope, dropout=0.1, moe_hidden=2048, num_experts=4, top_k=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = FlashMultiheadAttention(embed_dim, num_heads, dropout=dropout, rope=rope)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.moe = MoEFeedForward(embed_dim, moe_hidden, num_experts=num_experts, top_k=top_k, dropout=dropout)

    def forward(self, x, cache=None, use_cache=False, layer_idx=0):
        attn_in = self.norm1(x)
        attn_cache = None if cache is None else cache.get(layer_idx)
        attn_out, new_cache = self.attn(attn_in, cache=attn_cache, use_cache=use_cache, layer_idx=layer_idx)
        x = x + attn_out
        moe_in = self.norm2(x)
        x = x + self.moe(moe_in)
        if use_cache:
            if cache is None:
                cache = {}
            cache[layer_idx] = new_cache
        return x, cache


class VisionInspiredFusionTransformerLLM(nn.Module):
    def __init__(self, static_dim, dynamic_dim, graphical_dim, network_dim,
                 embed_dim=512, depth=12, n_heads=16, dropout=0.1, num_classes=2,
                 num_global_tokens=4, num_experts=4, top_k=2, rope_base=10000):
        super().__init__()
        self.embed_dim = embed_dim
        self.dropout = nn.Dropout(dropout)
        self.num_global_tokens = num_global_tokens
        self.modality_order = ['static', 'dynamic', 'graphical', 'network']
        patch_config = {
            'static': {'dim': static_dim, 'patch': 64},
            'dynamic': {'dim': dynamic_dim, 'patch': 48},
            'graphical': {'dim': graphical_dim, 'patch': 96},
            'network': {'dim': network_dim, 'patch': 64},
        }
        self.patchers = nn.ModuleDict({name: PatchEmbed(cfg['patch'], embed_dim) for name, cfg in patch_config.items()})
        self.lora_adapters = nn.ModuleDict({name: LoRAAdapter(embed_dim) for name in self.modality_order})
        self.modality_adapters = nn.ModuleDict({
            name: nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim),
                nn.GELU(),
                nn.Linear(embed_dim, embed_dim),
            ) for name in self.modality_order
        })
        self.global_tokens = nn.Parameter(torch.randn(1, num_global_tokens, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.rope = RotaryEmbedding(embed_dim // n_heads, base=rope_base)
        self.blocks = nn.ModuleList([
            TransformerBlock(
                embed_dim,
                n_heads,
                rope=self.rope,
                dropout=dropout,
                moe_hidden=embed_dim * 4,
                num_experts=num_experts,
                top_k=top_k,
            )
            for _ in range(depth)
        ])
        self.retrieval_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
        )
        self.classifier = nn.Linear(embed_dim, num_classes)
        self.rationale_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, len(self.modality_order))
        )
        self.speculative_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, len(self.modality_order))
        )

    def _prepare_batch(self, static, dynamic, graphical, network):
        if isinstance(static, dict):
            return static
        return {
            'static': static,
            'dynamic': dynamic,
            'graphical': graphical,
            'network': network,
        }

    def forward(self, static, dynamic=None, graphical=None, network=None, cache=None, use_cache=False):
        batch = self._prepare_batch(static, dynamic, graphical, network)
        tokens = []
        for modality in self.modality_order:
            feats = batch[modality]
            if feats.dim() == 1:
                feats = feats.unsqueeze(0)
            patches = self.patchers[modality](feats)
            patches = self.lora_adapters[modality](patches)
            patches = self.modality_adapters[modality](patches)
            tokens.append(patches)
        tokens = torch.cat(tokens, dim=1)
        B = tokens.shape[0]
        global_tokens = self.global_tokens.repeat(B, 1, 1)
        x = torch.cat([global_tokens, tokens], dim=1)
        x = x + self.pos_embed[:, : x.shape[1], :]
        layer_cache = cache if use_cache else None
        for layer_idx, block in enumerate(self.blocks):
            x, layer_cache = block(x, cache=layer_cache, use_cache=use_cache, layer_idx=layer_idx)
        pooled = self.retrieval_head(x[:, : self.num_global_tokens].mean(dim=1))
        logits = self.classifier(pooled)
        rationale = torch.softmax(self.rationale_head(pooled), dim=-1)
        speculative = torch.softmax(self.speculative_head(pooled), dim=-1)
        return logits, rationale, speculative

    def explain(self, static, dynamic, graphical, network):
        self.eval()
        with torch.no_grad():
            _, rationale, speculative = self.forward(static, dynamic, graphical, network)
        return rationale, speculative

    def regularization_loss(self):
        return torch.tensor(0.0, device=self.global_tokens.device)


primary_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if primary_device.type == 'cuda':
    torch.cuda.set_device(0)
model = VisionInspiredFusionTransformerLLM(
    static_dim, dynamic_dim, graphical_dim, network_dim, num_classes=NUM_CLASSES
).to(primary_device)
available_gpus = torch.cuda.device_count() if primary_device.type == 'cuda' else 0
if available_gpus >= 2:
    model = nn.DataParallel(model, device_ids=[0, 1])

print('Model parameters (M):', sum(p.numel() for p in model.parameters()) / 1e6)


Model parameters (M): 116.028474


In [25]:
# Training utilities
def core_model():
    return model.module if isinstance(model, nn.DataParallel) else model

class_counts = np.bincount(y, minlength=NUM_CLASSES)
class_weights = torch.tensor(
    len(y) / (NUM_CLASSES * class_counts),
    dtype=torch.float32,
    device=primary_device,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(core_model().parameters(), lr=9e-4, weight_decay=1e-4, betas=(0.9, 0.98))
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=3, T_mult=2)
reg_coef = 2e-4
amp_enabled = primary_device.type == 'cuda'
scaler = torch.amp.GradScaler(device='cuda') if amp_enabled else None
grad_clip = 1.0

def run_epoch(loader, train=True):
    phase = 'train' if train else 'eval'
    model.train(train)
    if not train:
        model.eval()
    losses = []
    preds = []
    trues = []
    pbar = tqdm(loader, desc=phase)
    for step, batch in enumerate(pbar):
        static = batch['static'].to(primary_device, non_blocking=True)
        dynamic = batch['dynamic'].to(primary_device, non_blocking=True)
        graphical = batch['graphical'].to(primary_device, non_blocking=True)
        network = batch['network'].to(primary_device, non_blocking=True)
        labels = batch['label'].to(primary_device, non_blocking=True)

        context = torch.autocast(device_type='cuda', dtype=torch.float16) if amp_enabled else nullcontext()
        with context:
            logits, _, _ = model(static, dynamic, graphical, network)
            base_loss = criterion(logits, labels)
            loss = base_loss + reg_coef * core_model().regularization_loss()
        if train:
            optimizer.zero_grad(set_to_none=True)
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(core_model().parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(core_model().parameters(), grad_clip)
                optimizer.step()
            scheduler.step(step + 1)

        losses.append(loss.item())
        batch_preds = logits.detach().cpu().numpy().argmax(axis=1)
        preds.extend(batch_preds.tolist())
        trues.extend(labels.detach().cpu().numpy().tolist())

    acc = accuracy_score(trues, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(trues, preds, average='binary', zero_division=0)
    return np.mean(losses), acc, precision, recall, f1

# Training loop
n_epochs = 8
best_val_f1 = 0.0
for epoch in range(1, n_epochs + 1):
    start = time.time()
    train_loss, train_acc, train_prec, train_rec, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_prec, val_rec, val_f1 = run_epoch(val_loader, train=False)
    elapsed = time.time() - start
    print(
        f'Epoch {epoch:02d} | Train loss {train_loss:.4f} acc {train_acc:.3f} f1 {train_f1:.3f} | '
        f'Val loss {val_loss:.4f} acc {val_acc:.3f} f1 {val_f1:.3f} | {elapsed:.1f}s'
    )
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(core_model().state_dict(), 'best_multimodal_model_llm.pt')
        print('Saved best model with F1:', best_val_f1)


eval: 100%|██████████| 7/7 [00:00<00:00,  8.53it/s]


Epoch 01 | Train loss 0.8003 acc 0.502 f1 0.531 | Val loss 0.7208 acc 0.500 f1 0.000 | 5.1s


eval: 100%|██████████| 7/7 [00:00<00:00,  8.66it/s]


Epoch 02 | Train loss 0.6973 acc 0.500 f1 0.562 | Val loss 0.7021 acc 0.500 f1 0.000 | 5.0s


eval: 100%|██████████| 7/7 [00:00<00:00,  8.73it/s]



...

Epoch 06 | Train loss 0.6938 acc 0.498 f1 0.466 | Val loss 0.6942 acc 0.500 f1 0.667 | 5.1s


eval: 100%|██████████| 7/7 [00:00<00:00,  9.26it/s]



Epoch 07 | Train loss 0.6936 acc 0.499 f1 0.561 | Val loss 0.6945 acc 0.500 f1 0.000 | 4.7s


eval: 100%|██████████| 7/7 [00:00<00:00,  8.61it/s]

Epoch 08 | Train loss 0.6939 acc 0.500 f1 0.444 | Val loss 0.6928 acc 0.534 f1 0.664 | 5.1s


In [26]:
# Evaluation on test set with probabilities / ROC AUC
state_dict = torch.load('best_multimodal_model_llm.pt', map_location=primary_device)
core_model().load_state_dict(state_dict)
model.eval()
all_preds = []
all_probs = []
all_trues = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='test'):
        static = batch['static'].to(primary_device, non_blocking=True)
        dynamic = batch['dynamic'].to(primary_device, non_blocking=True)
        graphical = batch['graphical'].to(primary_device, non_blocking=True)
        network = batch['network'].to(primary_device, non_blocking=True)
        labels = batch['label'].to(primary_device, non_blocking=True)
        logits, _, _ = model(static, dynamic, graphical, network)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_trues.extend(labels.cpu().numpy().tolist())

acc = accuracy_score(all_trues, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_trues, all_preds, average='binary', zero_division=0)
try:
    auc = roc_auc_score(all_trues, all_probs)
except Exception:
    auc = float('nan')
print(f'Test Acc {acc:.4f} Prec {prec:.4f} Rec {rec:.4f} F1 {f1:.4f} AUC {auc:.4f}')

sample_batch = next(iter(test_loader))
refined_scores, speculative_scores = core_model().explain(
    sample_batch['static'][:1].to(primary_device),
    sample_batch['dynamic'][:1].to(primary_device),
    sample_batch['graphical'][:1].to(primary_device),
    sample_batch['network'][:1].to(primary_device),
)
refined = refined_scores.squeeze().cpu().numpy()
speculative = speculative_scores.squeeze().cpu().numpy()
modalities = core_model().modality_order
reason_map = {name: float(refined[idx]) for idx, name in enumerate(modalities)}
spec_map = {name: float(speculative[idx]) for idx, name in enumerate(modalities)}
print('Refined reasoning weights:', reason_map)
print('Speculative draft weights:', spec_map)

reason_text = {
    'static': 'Static metadata & headers indicating packed/obfuscated binaries.',
    'dynamic': 'Behavioral telemetry spikes in dynamic traces.',
    'graphical': 'CFG/API structure suggesting malicious control flow.',
    'network': 'Network beacons and traffic anomalies.',
}
print('Reasoning score interpretation:')
for name, score in sorted(reason_map.items(), key=lambda kv: kv[1], reverse=True):
    meaning = reason_text.get(name, 'Modality evidence contributing to the decision.')
    speculative_hint = spec_map.get(name, 0.0)
    print(f"- {name}: refined {score:.2%} | draft {speculative_hint:.2%} => {meaning}")

def reasoning_to_text(reason_dict):
    sorted_items = sorted(reason_dict.items(), key=lambda kv: kv[1], reverse=True)
    phrases = [f"{name} ({score:.0%})" for name, score in sorted_items]
    return 'Top evidence: ' + ', '.join(phrases)

print('Tokenizer-ready rationale:')
print(reasoning_to_text(reason_map))


test: 100%|██████████| 8/8 [00:01<00:00,  6.53it/s]

Test Acc 0.5000 Prec 0.5000 Rec 1.0000 F1 0.6667 AUC 0.5575


Refined reasoning weights: {'static': 0.22877219319343567, 'dynamic': 0.21380376815795898, 'graphical': 0.19778083264827728, 'network': 0.35964325070381165}
Speculative draft weights: {'static': 0.1374143809080124, 'dynamic': 0.47394442558288574, 'graphical': 0.1844920814037323, 'network': 0.2041490226984024}
Reasoning score interpretation:
- network: refined 35.96% | draft 20.41% => Network beacons and traffic anomalies.
- static: refined 22.88% | draft 13.74% => Static metadata & headers indicating packed/obfuscated binaries.
- dynamic: refined 21.38% | draft 47.39% => Behavioral telemetry spikes in dynamic traces.
- graphical: refined 19.78% | draft 18.45% => CFG/API structure suggesting malicious control flow.
Tokenizer-ready rationale:
Top evidence: network (36%), static (23%), dynamic (21%), graphical (20%)


**Next steps & notes**
- Replace synthetic data with real feature extraction pipelines for each modality.
- Tune encoder architectures per modality (e.g., CNN for raw images, Transformer/text encoder for dynamic traces).
- Add data augmentation, class weighting, and more robust evaluation (cross-validation).
- Consider modality missingness handling (mask tokens) and interpretability (attention visualization).

In [27]:
total_params = sum(p.numel() for p in core_model().parameters())
trainable_params = sum(p.numel() for p in core_model().parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,} (~{total_params / 1e6:.2f}M)')
print(f'Trainable parameters: {trainable_params:,} (~{trainable_params / 1e6:.2f}M)')

Total parameters: 116,028,474 (~116.03M)
Trainable parameters: 116,028,474 (~116.03M)
